# Optimize LP01 input–output preservation: mode 1 fixed at 100 kW, total power variable

This is the D_IO version of `step_index_100kw_power_distri.ipynb`. The `100kw`
filename identifies its source: **100 kW is the fixed mode-1 peak power**.
Modes 2 and 3 each vary independently from 0 to 100 kW, so total input peak power
varies from 100 to 300 kW. Both relative phases are adjustable:

$$P=[100000,P_2,P_3]\;\mathrm{W},\qquad
\phi=[0,\phi_2,\phi_3].$$

All modes share the same Gaussian width and timing. The launch is constructed as
`gaussian_pulse(domain, 3; peak_power=sum(powers_W), fwhm=0.2,
coefficients=sqrt.(powers_W./powers_W[1]).*cis.(phases_rad))`.
This produces the specified **physical modal peak powers**, rather than fixing
the total power while changing normalized coefficients.

Minimize the complex input–nonlinear-output distortion of **mode 1 (LP01) only**.
All three modes still propagate, but only LP01 enters the overlap and its two norms:

$$F_{\rm IO}=\frac{\left|\int A_1^*(0,t)A_1(L,t)\,dt\right|^2}
{\left(\int |A_1(0,t)|^2dt\right)
\left(\int |A_1(L,t)|^2dt\right)},\qquad D_{\rm IO}=1-F_{\rm IO}.$$

The input and output use the same time grid. There is no temporal alignment or
phase-alignment search. The squared complex overlap is naturally invariant to a
single global phase. Relative launch phases can still affect LP01 through the
three-mode nonlinear propagation; changes confined to the other output modes
do not enter this objective. The baseline is
**[100, 50, 50] kW with all phases zero**. Only nonlinear propagation is required
for an objective call; a linear branch is computed for the two final diagnostics.

In [ ]:
using Pkg
workspace = let dir = abspath(@__DIR__)
    while !isfile(joinpath(dir, "notebook_helpers", "run_library.py"))
        dirname(dir) == dir && error("Open this notebook inside the Nonlinear Optics workspace.")
        dir = dirname(dir)
    end
    dir
end
ROOT = get(ENV, "DIO_PROJECT_ROOT", get(ENV, "DNL_PROJECT_ROOT", joinpath(workspace, "FastDifferentiableMMNLO")))
Pkg.activate(ROOT)
using PulsePropagation, FFTW, LinearAlgebra, Printf, TOML, Dates, SHA, NPZ, Serialization
using PyPlot
BLAS.set_num_threads(1)
FFTW.set_num_threads(1)
run_dir = haskey(ENV, "DIO_RUN_DIR") ? ENV["DIO_RUN_DIR"] :
    haskey(ENV, "DNL_RUN_DIR") ? ENV["DNL_RUN_DIR"] :
    mktempdir(mkpath(joinpath(workspace, "output", "organized", "runs", "Modal powers and relative phases", "Gradient descent")); prefix=Dates.format(now(), "yyyy-mm-dd") * "_dio_100kw_power_distri_", cleanup=false)
mkpath(run_dir)
plot_settings_path = joinpath(workspace, "notebook_helpers", "julia", "plot_settings.toml")
println("Results: ", run_dir)

# Load the launch, objective, descent, and plotting functions used below.
include(joinpath(workspace, "notebook_helpers", "julia", "Modal powers and relative phases", "shared.jl"))
include(joinpath(workspace, "notebook_helpers", "julia", "Modal powers and relative phases", "dio_100kw_power_distri.jl"))

## 1. Four controls and numerical settings

Use coordinates $x=(q_2,q_3,\phi_2,\phi_3)$ with $q_j=P_j/100000$.
Thus $dD/dq_j=100000\,dD/dP_j$. Power derivatives use 500 W perturbations
($h_q=0.005$); phase derivatives use 0.01 rad. Use a central difference in the
interior and a second-order one-sided formula near a power bound. At zero power,
the phase of that empty mode has no physical effect; the one-sided power
derivative is a finite-resolution estimate because field amplitude scales as
$\sqrt{P}$.

Power steps are projected onto [0, 100] kW, phases are wrapped to [-π, π), and
one iteration moves at most 10 kW and 0.1 rad per control. Armijo backtracking
requires decreasing D_IO and at least half the baseline's fine-grid **LP01
output photons**. Stop after 20 iterations or 200 new optimizer evaluations.
Use 5 mm propagation for the search and 1 mm for baseline/selected validation;
refine the search step only if the essential baseline accuracy check fails.

In [ ]:
fixed_mode1_power_W = 100000.0
baseline_x = [0.5, 0.5, 0.0, 0.0]
fd_steps = [0.005, 0.005, 0.01, 0.01]
learning_rate = 0.10
max_iterations = 20
max_optimizer_evaluations = 200
max_coordinate_changes = [0.10, 0.10, 0.10, 0.10]
gradient_tolerance = 1e-4
minimum_brightness_fraction = 0.50
search_dz = 0.005
validation_dz = 0.001
dio_step_tolerance = 2e-4
sweep_step_W = 2000.0
settings = Dict(
    "control"=>"q2=P2/100000, q3=P3/100000, phi2_rad, phi3_rad",
    "objective"=>"D_IO=1-F_IO: normalized squared complex input/nonlinear-output overlap, LP01 only in numerator and both norms",
    "fixed_mode1_power_W"=>fixed_mode1_power_W,
    "baseline_powers_W"=>powers_from_x(baseline_x),
    "baseline_phases_rad"=>phases_from_x(baseline_x),
    "mode2_mode3_bounds_W"=>[0.0, 100000.0], "total_power_bounds_W"=>[100000.0, 300000.0],
    "finite_difference_power_W"=>500.0, "finite_difference_phase_rad"=>0.01,
    "learning_rate"=>learning_rate, "max_iterations"=>max_iterations,
    "max_optimizer_evaluations"=>max_optimizer_evaluations,
    "max_power_change_W"=>10000.0, "max_phase_change_rad"=>0.1,
    "projected_gradient_tolerance"=>gradient_tolerance,
    "minimum_brightness_fraction"=>minimum_brightness_fraction,
    "brightness_definition"=>"LP01 output photons relative to LP01 fine-baseline photons",
    "initial_search_dz_m"=>search_dz, "search_dz_m"=>search_dz,
    "validation_dz_m"=>validation_dz, "dio_step_tolerance"=>dio_step_tolerance,
    "conditional_power_sweep_step_W"=>sweep_step_W,
    "Nt"=>2048, "time_window_ps"=>10.0, "pulse_fwhm_ps"=>0.2,
    "fiber_length_m"=>2.86, "carrier_m"=>1379.2e-9,
    "core_radius_m"=>52.5e-6, "NA"=>0.1, "num_modes"=>3,
    "objective_modes"=>[1], "objective_filter"=>"none",
    "temporal_alignment"=>false, "phase_alignment_search"=>false,
    "linear_branch"=>"baseline and selected diagnostic plots only; n2=0",
    "cp_target_error"=>1e-3, "cp_seed"=>Int(0x4d4d474e),
    "julia_version"=>string(VERSION),
    "manifest_sha256"=>bytes2hex(sha256(read(joinpath(ROOT, "Manifest.toml"))))
)
save_settings()
@printf("Mode 1 fixed: %.1f kW; total input range: 100–300 kW\n", fixed_mode1_power_W/1000)
println("Baseline powers (W): ", powers_from_x(baseline_x), "; phases (rad): ", phases_from_x(baseline_x))

## 2. Same fiber, three propagated modes

Mode 1 is LP01, mode 2 is LP11-cos, and mode 3 is LP11-sin. Keep the
1379.2 nm carrier, 52.5 μm core radius, NA 0.1, 2.86 m length, 0.2 ps FWHM,
and 2048 time samples. Each candidate launches all nonzero requested powers.

In [ ]:
time_grid = TimeGrid(settings["Nt"], settings["time_window_ps"])
domain = MMGNLSEDomain(degrees_of_freedom(:time, :space), time_grid)
properties = compute_fiber_properties(
    StepIndex(settings["carrier_m"], settings["core_radius_m"], settings["NA"]),
    Silica(), settings["num_modes"]; polarization=:scalar, mode_basis=:LP, beta_order=5)
S_cp = cp_compress(properties.S; error=settings["cp_target_error"], seed=settings["cp_seed"])
parameters_cp = MMGNLSEParameters(domain; length=settings["fiber_length_m"],
    beta=properties.beta, S=S_cp, n2=properties.n2, omega0=properties.omega0, raman=properties.raman)
parameters_linear = MMGNLSEParameters(domain; length=settings["fiber_length_m"],
    beta=properties.beta, S=S_cp, n2=0.0, omega0=properties.omega0, raman=properties.raman)
LP01_photon_observable = SpectralPhotonNumber(filter=ones(settings["Nt"]), modes=[1])
all_photons = SpectralPhotonNumber(filter=ones(settings["Nt"]), modes=:all)
# Purely algebraic launch normalization and relative-phase sanity; no propagation.
for x in (baseline_x, [0.0,1.0,0.4,-0.7], [1.0,0.2,-1.1,2.2])
    a = construct_input(x)
    k = argmax(vec(sum(abs2.(a); dims=(2,3))))
    @assert isapprox(vec(abs2.(a[k,:,1])), powers_from_x(x); rtol=1e-12, atol=1e-9)
    expected = sqrt.(powers_from_x(x)./fixed_mode1_power_W).*cis.(phases_from_x(x))
    @assert isapprox(vec(a[k,:,1])./a[k,1,1], expected; rtol=1e-12, atol=1e-12)
end
println("Launch powers and relative phases match the requested construction.")
println("Mode labels: ", properties.modes.labels)
@printf("CP relative error: %.3g\n", S_cp.relative_error)

## 3. D_IO evaluation and finite differences

Each new `dio_at(x)` call performs **one nonlinear solve**. Select LP01 before
forming the complex overlap and both denominator norms. The common
time spacing cancels from the normalized overlap. Cache both scalar results and
complex fields on disk, so later sweeps, diagnostics, and reruns reuse them.
There is no linear solve in the objective or finite differences.

In [ ]:
evaluation_cache = Dict{String,NamedTuple}()
field_cache = Dict{String,NamedTuple}()
evaluations = NamedTuple[]
cache_dir = joinpath(run_dir, "objective_cache")
mkpath(cache_dir)
physics_signature = bytes2hex(sha256(string("D_IO_LP01_v2", settings["Nt"], settings["time_window_ps"],
    settings["pulse_fwhm_ps"], settings["fiber_length_m"], settings["carrier_m"],
    settings["core_radius_m"], settings["NA"], settings["num_modes"], settings["cp_target_error"],
    settings["cp_seed"], settings["manifest_sha256"])))
let a = zeros(ComplexF64, 4, 3, 1)
    a[1,1,1]=1; a[1,2,1]=1
    @assert isapprox(field_fidelity(a,2cis(0.7).*a),1.0; atol=1e-12)
    b=copy(a); b[:,2,:].*=-1; b[1,3,1]=3
    @assert isapprox(field_fidelity(a,b),1.0; atol=1e-12) # Other modes do not enter F_IO.
    b=copy(a); b[1,1,1]=0; b[2,1,1]=1
    @assert field_fidelity(a,b)==0.0 # LP01 temporal distortion changes F_IO.
end
println("LP01 overlap algebra, other-mode invariance, and objective range checks passed.")

## 4. One fine baseline check

Compare the search baseline against the 1 mm baseline and set the photon floor
from the fine result. A failed propagation-step comparison triggers only the
necessary baseline step refinement. There is no repeated half-gradient,
mode-count, time-grid, or tensor-comparison campaign.

In [ ]:
baseline = dio_at(baseline_x)
baseline_fine = dio_at(baseline_x; dz=validation_dz)
baseline_step_error = abs(baseline.D_IO-baseline_fine.D_IO)
baseline_refinements=0
while baseline_step_error>dio_step_tolerance && search_dz>validation_dz
    global search_dz=max(validation_dz,search_dz/2)
    global baseline=dio_at(baseline_x;dz=search_dz)
    global baseline_step_error=abs(baseline.D_IO-baseline_fine.D_IO)
    global baseline_refinements+=1
end
brightness_floor=minimum_brightness_fraction*baseline_fine.LP01_photons
settings["search_dz_m"]=search_dz
settings["baseline_step_refinements"]=baseline_refinements
save_settings()
@assert baseline_step_error<=dio_step_tolerance
@printf("Baseline D_IO: search %.9f; fine %.9f; difference %.3g\n",baseline.D_IO,baseline_fine.D_IO,baseline_step_error)
@printf("Search dz %.3f mm; LP01 photon floor %.6e\n",search_dz*1000,brightness_floor)

## 5. Projected gradient descent with both phases free

Propose $x_{\rm new}=x-\eta\nabla D$ with component step caps, project powers,
and wrap phases. Use the shortest wrapped phase increment in the Armijo test.
Accepted states must reduce D_IO and satisfy brightness. Only accepted steps
enter the history. Budget exhaustion or a failed line search is reported as
such; it does not certify a local or global minimum.

In [ ]:
optimization=optimize_launch()
println("Stop: ",optimization.status)
println("Selected modal powers (W): ",powers_from_x(x_from_row(optimization.best)))
println("Selected relative phases (rad): ",phases_from_x(x_from_row(optimization.best)))

## 6. One fine selected-launch check

Evaluate the selected launch at 1 mm and compare with the cached search result.
Report its signed improvement, photon retention, propagation-step discrepancy,
and decreasing accepted history. An improvement is reported only if observed.
No neighbor campaign or claim of a certified minimum is made.

In [ ]:
best_x=x_from_row(optimization.best)
best_fine=dio_at(best_x;dz=validation_dz)
improvement_D_IO=baseline_fine.D_IO-best_fine.D_IO
best_step_error=abs(optimization.best.D_IO-best_fine.D_IO)
brightness_passed=best_fine.LP01_photons>=brightness_floor
decreasing_accepts=all(diff([r.D_IO for r in optimization.history]).<0)
step_check_passed=max(baseline_step_error,best_step_error)<=dio_step_tolerance
@assert brightness_passed "Selected fine result fails the brightness floor."
@assert decreasing_accepts "Accepted history is not decreasing."
summary=Dict(
    "status"=>optimization.status,"fixed_mode1_power_W"=>fixed_mode1_power_W,
    "baseline_powers_W"=>powers_from_x(baseline_x),"baseline_phases_rad"=>phases_from_x(baseline_x),
    "selected_powers_W"=>powers_from_x(best_x),"selected_phases_rad"=>phases_from_x(best_x),
    "baseline_D_IO"=>baseline_fine.D_IO,"selected_D_IO"=>best_fine.D_IO,
    "baseline_F_IO"=>baseline_fine.F_IO,"selected_F_IO"=>best_fine.F_IO,
    "signed_improvement_D_IO"=>improvement_D_IO,
    "relative_D_IO_reduction"=>improvement_D_IO/baseline_fine.D_IO,
    "brightness_ratio"=>best_fine.LP01_photons/baseline_fine.LP01_photons,
    "baseline_LP01_photons"=>baseline_fine.LP01_photons,"selected_LP01_photons"=>best_fine.LP01_photons,
    "baseline_total_photons"=>baseline_fine.total_photons,"selected_total_photons"=>best_fine.total_photons,
    "objective_mode"=>"LP01 (mode 1) only", "brightness_metric"=>"LP01 output photons",
    "baseline_step_error"=>baseline_step_error,"selected_step_error"=>best_step_error,
    "search_dz_m"=>search_dz,"validation_dz_m"=>validation_dz,
    "brightness_passed"=>brightness_passed,"decreasing_accepts"=>decreasing_accepts,
    "step_check_passed"=>step_check_passed,"improvement_observed"=>improvement_D_IO>0,
    "accepted_steps"=>length(optimization.history)-1,
    "new_optimizer_objective_evaluations"=>optimization.new_objective_evaluations,
    "optimization_seconds"=>optimization.seconds,"cp_relative_error"=>S_cp.relative_error,
    "selected_output_input_energy_ratio"=>best_fine.output_input_energy_ratio,
    "validation_scope"=>"three-mode fixed model; launch algebra; LP01-only D_IO; LP01 photon brightness; decreasing accepts; one fine baseline/selected check"
)
save_summary()
@printf("Fine D_IO: %.9f → %.9f; signed change %.9f\n",baseline_fine.D_IO,best_fine.D_IO,improvement_D_IO)
@printf("LP01 photons retained: %.3f%%; selected step discrepancy %.3g\n",100summary["brightness_ratio"],best_step_error)
println("Fine-step check passed: ",step_check_passed)
@assert step_check_passed "Essential propagation-step check failed; refine search_dz before interpreting the search."

## 7. Conditional power slices and the actual descent path

Sweep $P_2=0:2000:100000$ W while holding $P_3,\phi_2,\phi_3$ at the
selected values, then sweep $P_3$ with the other controls selected. These are
**conditional one-dimensional slices**, not a two-dimensional landscape.
Each uncached point performs one nonlinear solve. The slice minimum does not
replace the descent result. Plot the actual power path and phase histories
separately because the phases change during descent.

In [ ]:
scan_powers_W=collect(0.0:sweep_step_W:100000.0)
scan2=[dio_at([p/fixed_mode1_power_W,best_x[2],best_x[3],best_x[4]]) for p in scan_powers_W]
scan3=[dio_at([best_x[1],p/fixed_mode1_power_W,best_x[3],best_x[4]]) for p in scan_powers_W]
save_rows("conditional_P2_sweep_D_IO.csv",scan2)
save_rows("conditional_P3_sweep_D_IO.csv",scan3)
summary["conditional_sweep_points"]=length(scan2)+length(scan3)
for (label,scan) in (("P2",scan2),("P3",scan3))
    feasible=filter(r->r.LP01_photons>=brightness_floor,scan)
    if !isempty(feasible)
        s=feasible[argmin([r.D_IO for r in feasible])]
        summary[label*"_slice_best_D_IO"]=s.D_IO
        summary[label*"_slice_best_powers_W"]=powers_from_x(x_from_row(s))
        summary[label*"_slice_has_lower_feasible_sample"]=s.D_IO<optimization.best.D_IO
    end
end
save_summary()
fig,axes=subplots(1,2;figsize=(13.5,4.7),constrained_layout=true)
for (j,scan) in enumerate((scan2,scan3))
    ax=axes[j]
    ax.plot(scan_powers_W./1000,[r.D_IO for r in scan],"o-";color="#366b9e",markersize=3,label="Conditional slice")
    ax.scatter([best_x[j]*100],[optimization.best.D_IO];color="#cf7c32",marker="*",s=130,label="Selected search state")
    bad=findall(r->r.LP01_photons<brightness_floor,scan)
    if !isempty(bad)
        ax.scatter(scan_powers_W[bad]./1000,[scan[k].D_IO for k in bad];color="gray",marker="x",label="Below photon floor")
    end
    ax.set_xlabel("Mode $(j+1) input peak power (kW)"); ax.set_ylabel("D_IO (LP01 only)")
    fixed_power=j==1 ? best_fine.P3_W : best_fine.P2_W
    ax.set_title(@sprintf("P%d slice: other variable power %.3f kW\nphases fixed at (%.4f, %.4f) rad",j+1,fixed_power/1000,best_x[3],best_x[4]);fontsize=10)
    ax.legend(;fontsize=8);ax.grid(;alpha=0.2)
end
fig.suptitle(@sprintf("Mode 1 fixed at 100 kW; conditional power slices; dz %.3f mm",search_dz*1000))
savefig(joinpath(run_dir,"conditional_power_slices_D_IO.png");dpi=180)
savefig(joinpath(run_dir,"conditional_power_slices_D_IO.pdf"));display(fig);close(fig)

h=optimization.history
iterations=[r.iteration for r in h]
fig,axes=subplots(2,2;figsize=(13.5,9),constrained_layout=true)
axes[1,1].plot([r.P2_W/1000 for r in h],[r.P3_W/1000 for r in h],"o-";color="#cf7c32",markersize=4)
for i in 2:length(h)
    axes[1,1].annotate("";xy=(h[i].P2_W/1000,h[i].P3_W/1000),xytext=(h[i-1].P2_W/1000,h[i-1].P3_W/1000),arrowprops=Dict("arrowstyle"=>"-|>","color"=>"#cf7c32"))
end
axes[1,1].scatter([h[1].P2_W/1000],[h[1].P3_W/1000];facecolors="white",edgecolors="#cf7c32",s=85,label="Baseline")
axes[1,1].scatter([h[end].P2_W/1000],[h[end].P3_W/1000];color="#cf7c32",marker="*",s=130,label="Selected")
axes[1,1].set_xlabel("P2 (kW)");axes[1,1].set_ylabel("P3 (kW)");axes[1,1].set_title("Accepted power path; phases also change");axes[1,1].legend(;fontsize=8)
axes[1,2].plot(iterations,[r.D_IO for r in h],"o-";color="#cf7c32",markersize=4)
axes[1,2].scatter([0,h[end].iteration],[baseline_fine.D_IO,best_fine.D_IO];color="black",marker="x",label="1 mm checks")
axes[1,2].set_ylabel("D_IO (LP01 only)");axes[1,2].set_xlabel("Accepted iteration");axes[1,2].set_title("Stop: "*optimization.status);axes[1,2].legend(;fontsize=8)
for (field,label) in ((:phi2_rad,"phi2"),(:phi3_rad,"phi3"))
    axes[2,1].plot(iterations,[getproperty(r,field) for r in h],"o-";markersize=3,label=label)
end
axes[2,1].set_ylabel("Relative phase (rad)");axes[2,1].set_xlabel("Accepted iteration");axes[2,1].legend()
axes[2,2].plot(iterations,[r.total_power_W/1000 for r in h],"o-";color="#366b9e",markersize=3,label="Total input")
axes[2,2].set_ylabel("Total input peak power (kW)");axes[2,2].set_xlabel("Accepted iteration");axes[2,2].set_title("P1 stays at 100 kW")
for ax in axes;ax.grid(;alpha=0.2);ax.spines["top"].set_visible(false);ax.spines["right"].set_visible(false);end
savefig(joinpath(run_dir,"D_IO_launch_optimization.png");dpi=180)
savefig(joinpath(run_dir,"D_IO_launch_optimization.pdf"));display(fig);close(fig)
println("Conditional slice minima are diagnostics; the selected result remains the descent result.")

## 8. Each mode: input, linear output, nonlinear output

Compute the linear branch only for these baseline/selected diagnostic figures.
Time graphs show modal power in kW. Spectra show power per frequency bin in W
against **wavelength only**; they are not densities per nm. Save the full complex
fields. The title reports LP01-only D_IO between input and nonlinear output.

In [ ]:
times_ps = collect(time_axis(time_grid))
wavelengths_nm = collect(wavelength_axis(time_grid, properties.omega0)) .* 1e9
wavelength_indices = findall(isfinite.(wavelengths_nm) .& (wavelengths_nm .> 0))
wavelength_indices = wavelength_indices[sortperm(wavelengths_nm[wavelength_indices])]
mode_names = ("LP01", "LP11-cos", "LP11-sin")
curve_labels = ("Input", "Linear output (diagnostic)", "Nonlinear output")
curve_colors = ("#777777", "#cf7c32", "#366b9e")
curve_styles = (":", "--", "-")

plot_modal_output("baseline", baseline_fine)
plot_modal_output("selected", best_fine)

## 9. Spatial mode shapes and their energy fractions

The fiber fixes the LP01 and the two LP11 basis shapes. Their signed relative
fields are shown first. Modal energy fractions compare the baseline and selected
launch; each branch is normalized to its own total energy. These fraction bars
are distinct from the absolute intensity images below.

In [ ]:
x = properties.modes.x
y = properties.modes.y
modes = properties.modes.fields  # axes: y, x, mode
mode_names = ("LP01", "LP11 — cos orientation", "LP11 — sin orientation")

let fig = figure(; figsize=(18, 6), constrained_layout=true)
    for ii = 1:3
        ax = subplot(1, 3, ii)
        # Normalize amplitude only for displaying the fixed mode shapes.
        profile = real.(modes[:, :, ii])
        profile = profile ./ maximum(abs, profile)
        mesh = ax.pcolormesh(x.*1e6, y.*1e6, profile;
            cmap="seismic", vmin=-1, vmax=1, shading="auto", rasterized=true)
        ax.set_title("Mode $ii: "*mode_names[ii])
        ax.set_xlabel("x (μm)")
        ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
        theta = range(0, 2pi; length=181)
        ax.plot(settings["core_radius_m"].*1e6.*cos.(theta),
            settings["core_radius_m"].*1e6.*sin.(theta);
            color="#555555", linestyle="--", linewidth=0.8)
        fig.colorbar(mesh; ax, shrink=0.75, label="Relative field amplitude")
    end
    fig.suptitle("The three spatial modes — dashed line marks the fiber core")
    savefig(joinpath(run_dir, "spatial_modes.png"); dpi=160)
    savefig(joinpath(run_dir, "spatial_modes.pdf"))
    display(fig)
    close(fig)
end
npzwrite(joinpath(run_dir, "spatial_mode_fields.npz"),
    Dict("x_m"=>x, "y_m"=>y, "modes_y_x_m"=>modes))


fig, axes = subplots(1, 2; figsize=(12.5, 4.3), constrained_layout=true, sharey=true)
modal_rows = NamedTuple[]
for (case_index, (label, row)) in enumerate((("Baseline", baseline_fine), ("Selected", best_fine)))
    saved = field_cache[cache_key(x_from_row(row),row.dz)]
    for (j, a) in enumerate((saved.input, saved.linear, saved.nonlinear))
        energy = vec(sum(abs2.(a); dims=(1, 3)))
        fractions = energy./sum(energy)
        axes[case_index].bar(collect(1:3).+(j-2)*0.23, 100fractions;
            width=0.23, color=curve_colors[j], label=curve_labels[j])
        for m in 1:3
            push!(modal_rows, (; case=lowercase(label), branch=curve_labels[j],
                mode=m, energy_fraction=fractions[m]))
        end
    end
    axes[case_index].set_xticks(1:3, ["LP01", "LP11-cos", "LP11-sin"])
    axes[case_index].set_ylim(0, 105)
    axes[case_index].set_ylabel("Fraction of total pulse energy (%)")
    axes[case_index].set_title(@sprintf("%s: P=[100, %.3f, %.3f] kW",
        label,row.P2_W/1000,row.P3_W/1000))
    axes[case_index].legend(; fontsize=8)
    axes[case_index].grid(; axis="y", alpha=0.2)
end
save_rows("modal_energy_fractions.csv", modal_rows)
savefig(joinpath(run_dir, "baseline_selected_modal_fractions.png"); dpi=180)
savefig(joinpath(run_dir, "baseline_selected_modal_fractions.pdf"))
display(fig)
close(fig)

## 10. Actual baseline and selected beams

Reconstruct each beam by adding all three **complex** spatial mode fields before
squaring, retaining modal interference. Each panel uses its branch's own total
power peak and reports the corresponding time. Every absolute beam and modal
intensity plot reads the same `plot_settings.toml` beside this notebook. The
shared scale therefore gives the same intensity the same color across all D_IO
notebooks. Cached fields and beam maps support later redraws without simulation.

In [ ]:
plot_settings=TOML.parsefile(plot_settings_path)
beam_intensity_limits_W_per_um2=(Float64(plot_settings["beam_intensity_min_W_per_um2"]),Float64(plot_settings["beam_intensity_max_W_per_um2"]))
beam_colormap=plot_settings["colormap"]
beam_color_ticks=collect(range(beam_intensity_limits_W_per_um2...;length=6))


beam_cases = [("Baseline", baseline_fine), ("Selected", best_fine)]
beam_maps = [[peak_beam_map(a) for a in (
    field_cache[cache_key(x_from_row(row),row.dz)].input,
    field_cache[cache_key(x_from_row(row),row.dz)].linear,
    field_cache[cache_key(x_from_row(row),row.dz)].nonlinear)] for (_, row) in beam_cases]
observed_max = maximum(maximum(m.intensity) for maps in beam_maps for m in maps)
common_max = beam_intensity_limits_W_per_um2[2]
@assert observed_max <= common_max "Increase the shared intensity limit in plot_settings.toml."
fig, beam_axes = subplots(2, 3; figsize=(16, 9.3), constrained_layout=true)
meshes = []
for (case_index, (label, row)) in enumerate(beam_cases)
    for j in 1:3
        ax = beam_axes[case_index, j]
        m = beam_maps[case_index][j]
        mesh = ax.pcolormesh(x.*1e6, y.*1e6, m.intensity;
            cmap=beam_colormap, vmin=beam_intensity_limits_W_per_um2[1], vmax=common_max, shading="auto", rasterized=true)
        push!(meshes, mesh)
        ax.set_title(@sprintf("%s — %s\nP=[100, %.2f, %.2f] kW; peak %.2f kW at %.3f ps",
            label, curve_labels[j], row.P2_W/1000, row.P3_W/1000,
            m.peak_power_W/1000, m.peak_time_ps); fontsize=10)
        ax.set_xlabel("x (μm)")
        ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
    end
    maps = beam_maps[case_index]
    npzwrite(joinpath(run_dir, lowercase(label)*"_beam_maps.npz"), Dict(
        "x_um"=>x.*1e6, "y_um"=>y.*1e6,
        "input_W_per_um2"=>maps[1].intensity,
        "linear_W_per_um2"=>maps[2].intensity,
        "nonlinear_W_per_um2"=>maps[3].intensity,
        "peak_times_ps"=>[m.peak_time_ps for m in maps],
        "peak_powers_W"=>[m.peak_power_W for m in maps],
        "shared_color_min_W_per_um2"=>beam_intensity_limits_W_per_um2[1],
        "shared_color_max_W_per_um2"=>common_max))
end
fig.colorbar(meshes[end]; ax=vec(beam_axes), shrink=0.8, ticks=beam_color_ticks,
    label="Intensity at each pulse's peak (W/μm²)")
fig.suptitle("Baseline and selected beams: mode 1 fixed at 100 kW; total power variable")
savefig(joinpath(run_dir, "baseline_selected_beams.png"); dpi=160)
savefig(joinpath(run_dir, "baseline_selected_beams.pdf"))
display(fig)
close(fig)
println("All reconstructed spatial integrals match the modal peak powers.")

## 11. Individual nonlinear output-mode intensities

Each panel shows a single nonlinear output mode at its own modal-power peak.
The shapes are fixed by the fiber and weighted by the actual simulated modal
power. These separate intensities omit the cross-mode interference visible in
the combined beams. All six panels use the shared absolute intensity scale.

In [ ]:
modal_maps = NamedTuple[]
for (case_index, (label, row)) in enumerate(beam_cases)
    a = field_cache[cache_key(x_from_row(row),row.dz)].nonlinear
    fractions = [row.mode1_energy_fraction, row.mode2_energy_fraction, row.mode3_energy_fraction]
    for m in 1:3
        power_t = vec(sum(abs2.(a[:, m, :]); dims=2))
        k = argmax(power_t)
        intensity = abs2.(modes[:, :, m]) .* power_t[k] ./ 1e12
        push!(modal_maps, (; case_index, mode=m, label, intensity,
            peak_power_W=power_t[k], time_ps=times_ps[k], energy_fraction=fractions[m]))
    end
end
modal_observed_max = maximum(maximum(m.intensity) for m in modal_maps)
modal_common_max = beam_intensity_limits_W_per_um2[2]
@assert modal_observed_max <= modal_common_max "Increase the shared intensity limit in plot_settings.toml."
fig, modal_axes = subplots(2, 3; figsize=(16, 9.3), constrained_layout=true)
meshes = []
for m in modal_maps
    ax = modal_axes[m.case_index, m.mode]
    mesh = ax.pcolormesh(x.*1e6, y.*1e6, m.intensity;
        cmap=beam_colormap, vmin=beam_intensity_limits_W_per_um2[1], vmax=modal_common_max, shading="auto", rasterized=true)
    push!(meshes, mesh)
    ax.set_title(@sprintf("%s — mode %d (%s)\nenergy %.4g%%; peak %.4g kW at %.3f ps",
        m.label, m.mode, mode_names[m.mode], 100m.energy_fraction,
        m.peak_power_W/1000, m.time_ps); fontsize=10)
    ax.set_xlabel("x (μm)")
    ax.set_ylabel("y (μm)")
    ax.set_aspect("equal")
end
fig.colorbar(meshes[end]; ax=vec(modal_axes), shrink=0.8, ticks=beam_color_ticks,
    label="Single-mode peak intensity (W/μm²)")
fig.suptitle("Individual nonlinear output modes: baseline and selected launch")
savefig(joinpath(run_dir, "baseline_selected_output_modes.png"); dpi=160)
savefig(joinpath(run_dir, "baseline_selected_output_modes.pdf"))
display(fig)
close(fig)

npzwrite(joinpath(run_dir,"output_mode_maps.npz"),Dict(
    "x_um"=>x.*1e6,"y_um"=>y.*1e6,
    "intensities_W_per_um2"=>cat([m.intensity for m in modal_maps]...;dims=3),
    "peak_powers_W"=>[m.peak_power_W for m in modal_maps],
    "peak_times_ps"=>[m.time_ps for m in modal_maps],
    "energy_fractions"=>[m.energy_fraction for m in modal_maps]))
summary["largest_plotted_intensity_W_per_um2"]=max(observed_max,modal_observed_max)
summary["largest_beam_intensity_W_per_um2"]=observed_max
summary["largest_single_mode_intensity_W_per_um2"]=modal_observed_max
summary["shared_color_min_W_per_um2"]=beam_intensity_limits_W_per_um2[1]
summary["shared_color_max_W_per_um2"]=beam_intensity_limits_W_per_um2[2]
summary["total_new_objective_evaluations"]=length(evaluations)
save_summary()
@printf("Largest plotted intensity: %.9f W/μm²\n",summary["largest_plotted_intensity_W_per_um2"])
@printf("Completed: %d accepted steps; %d new optimizer evaluations; %d conditional sweep points.\n",
    length(optimization.history)-1,optimization.new_objective_evaluations,length(scan2)+length(scan3))
println("Stop: ",optimization.status)
println("Results: ",run_dir)